# TP 1 — Premier CNN sur MNIST

**Objectifs**

- Charger MNIST via `torchvision.datasets`.
- Construire un petit CNN à 2 blocs convolutifs.
- Écrire une boucle d'entraînement complète.
- Évaluer le modèle et visualiser quelques erreurs.

**Durée indicative : 60 minutes.**

> Note : on travaille sur un **sous-ensemble** de MNIST (3 000 train / 1 000 test) pour que tout tourne en quelques minutes sur CPU. Les valeurs d'accuracy seront un peu en-dessous de ce que donne MNIST complet (~99 %).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = "data"
print("device:", DEVICE)

## Exercice 1 — Charger MNIST en sous-ensemble

1. Crée un transform `ToTensor` (qui passe en `float32` dans `[0, 1]` et range en `(C, H, W)`).
2. Charge `datasets.MNIST` train et test (`download=True`).
3. Restreins le train aux 3 000 premiers exemples et le test aux 1 000 premiers (`Subset`).
4. Crée deux `DataLoader` avec `batch_size=64`, shuffle train uniquement.
5. Affiche 8 exemples du train avec leur label.

In [ ]:
# TODO


## Exercice 2 — Construire le CNN

Construis la classe suivante :

```python
class SmallCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        # conv1 : 1 → 16, kernel 3, padding 1
        # pool 2×2
        # conv2 : 16 → 32, kernel 3, padding 1
        # pool 2×2
        # fc1 : 32*7*7 → 64
        # fc2 : 64 → n_classes
        ...
    def forward(self, x):
        ...
```

Vérifie que `model(torch.zeros(1, 1, 28, 28)).shape == (1, 10)`.

<details>
<summary>💡 Coup de pouce</summary>

- `nn.Conv2d(in, out, kernel_size=3, padding=1)` conserve la taille spatiale.
- `nn.MaxPool2d(2)` divise H et W par 2 → après 2 pools sur 28×28, on est à 7×7.
- Pour le passage conv → FC : `x = x.flatten(start_dim=1)` aplatit en (B, 32*7*7).
- N'oublie pas `F.relu` entre les couches : `x = F.relu(self.conv1(x))`.

</details>

In [ ]:
# TODO


## Exercice 3 — Boucle d'entraînement

Écris une fonction `train_one_epoch(model, loader, opt, loss_fn)` qui parcourt le loader une fois, fait `forward / loss / backward / step`, et retourne la perte moyenne et l'accuracy.

Écris aussi `evaluate(model, loader, loss_fn)` qui calcule perte et accuracy en mode `eval` sans gradients.

Entraîne pendant 3 époques avec `Adam(lr=1e-3)` et `CrossEntropyLoss`. Affiche les courbes de perte et d'accuracy train / test.

<details>
<summary>💡 Coup de pouce</summary>

- Ordre canonique : `opt.zero_grad(); pred = model(x); loss = loss_fn(pred, y); loss.backward(); opt.step()`.
- Oublier `zero_grad()` = gradients accumulés entre batchs = bug subtil mais classique.
- En éval : `model.eval()` et `with torch.no_grad():` pour désactiver autograd et accélérer.
- Accuracy : `(pred.argmax(1) == y).float().mean().item()`.

</details>

In [ ]:
# TODO


## Exercice 4 — Visualiser les erreurs

Trouve les indices où la prédiction du modèle diffère du label, et affiche 6 erreurs avec un titre `(vrai → prédit)`. Quelles classes sont les plus confondues ?

<details>
<summary>💡 Coup de pouce</summary>

- Récupère prédictions sur tout le test : pour cela, accumule `preds.append(model(x).argmax(1))` dans la boucle d'éval.
- Indices erreurs : `wrong = (all_preds != all_labels).nonzero(as_tuple=True)[0]`.
- Affiche les 6 premières avec `plt.imshow(X_test[i].squeeze(), cmap='gray'); plt.title(f'{y_true[i]} → {y_pred[i]}')`.
- Typiquement MNIST : 4↔9, 3↔8, 7↔1.

</details>

In [ ]:
# TODO
